# 03 — I-JEPA Self-Supervised Pre-training

**Goal of this notebook:**
- Understand and implement the I-JEPA architecture (context encoder, target encoder, predictor)
- Pre-train a ViT-Small encoder on ALL LIDC-IDRI images **without labels**
- Save encoder checkpoints to Drive every 5 epochs so Colab disconnects don't lose progress
- Produce a loss curve and confirm representations are meaningful before moving to the probe


### 0. Keep-Alive (run this in a separate cell while training)

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade --force-reinstall optree

In [ ]:
# Keep-alive inutile en local (c'était pour Colab).
# Vous pouvez ignorer cette cellule.
print('Environnement local — pas de keep-alive nécessaire.')


### 1. Environment Setup

In [ ]:
# Pas de montage Google Drive — chemins locaux
print('Environnement local détecté — pas de montage nécessaire.')


In [ ]:
import subprocess, sys
pkgs = ['timm', 'einops', 'torch', 'torchvision', 'tqdm', 'matplotlib', 'scikit-learn']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('Installation terminée.')


In [ ]:
import os, sys

# Modifiez ROOT si nécessaire (par défaut : sous-dossier 'PFA' à côté du notebook)
ROOT      = os.environ.get('PROJECT_ROOT') or (os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()).lower() == 'notebooks' else os.getcwd()) 
os.environ['PROJECT_ROOT'] = ROOT
DATA_PROC  = os.path.join(ROOT, 'data', 'processed')
CKPT_DIR   = os.path.join(ROOT, 'checkpoints', 'ijepa')
FIGURES    = os.path.join(ROOT, 'results', 'figures', 'pretrain')
SRC        = os.path.join(ROOT, 'src')

for d in [CKPT_DIR, FIGURES]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, SRC)

# Quick prerequisite check
for f in ['config.json', 'train.csv', 'val.csv', 'test.csv']:
    p = os.path.join(DATA_PROC, f)
    status = 'OK' if os.path.exists(p) else 'MISSING — run 02_preprocessing.ipynb first'
    print(f'  {f}: {status}')


### 2. Imports & Hyperparameters

In [ ]:
!pip install optree

In [ ]:
import json, copy, math, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as T
import timm

# ── reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True   # faster conv on fixed-size inputs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── load preprocessing config
with open(os.path.join(DATA_PROC, 'config.json')) as f:
    cfg = json.load(f)

MEAN = cfg['mean']
STD  = cfg['std']
print(f'Dataset mean : {MEAN}')
print(f'Dataset std  : {STD}')


In [ ]:
# ── I-JEPA Hyperparameters  ─────────────────────
IMG_SIZE        = 224       # ViT input size
PATCH_SIZE      = 16        # each patch = 16x16 px  →  14x14 = 196 patches
N_PATCHES       = (IMG_SIZE // PATCH_SIZE) ** 2    # 196

EMBED_DIM       = 384       # ViT-Small hidden dim
PREDICTOR_DIM   = 192       # predictor bottleneck (half of embed)
PREDICTOR_DEPTH = 4         # number of transformer layers in predictor

# masking
TARGET_MASK_RATIO = 0.75    # fraction of patches predicted (per I-JEPA paper)
N_TARGET_BLOCKS   = 4       # number of rectangular target blocks per image
CONTEXT_SCALE     = (0.85, 1.0)  # context block covers 85-100% of image

# training
BATCH_SIZE     = 32        # reduce to 32 if CUDA OOM
NUM_WORKERS    = 2
NUM_EPOCHS     = 100
LR             = 1e-3
WEIGHT_DECAY   = 0.04
EMA_MOMENTUM   = 0.996      # target encoder exponential moving average
WARMUP_EPOCHS  = 10         # linear LR warmup

SAVE_EVERY     = 5          # save checkpoint every N epochs
RESUME_CKPT    = f'{CKPT_DIR}/ijepa_epoch_090.pth'  # None pour repartir de zéro       # set to a .pth path to resume, e.g. f'{CKPT_DIR}/ijepa_epoch_050.pth'

print(f'N_PATCHES      : {N_PATCHES}')
print(f'Target patches : ~{int(N_PATCHES * TARGET_MASK_RATIO)} / {N_PATCHES} per image')
print(f'Batch size     : {BATCH_SIZE}')
print(f'Epochs         : {NUM_EPOCHS}')
print(f'LR             : {LR}  (with {WARMUP_EPOCHS}-epoch warmup)')


### 3. I-JEPA Architecture

I-JEPA has three components:

| Component | What it does | Updated how |
|-----------|-------------|-------------|
| **Context encoder** | ViT-Small that sees the *visible* patches | Gradient descent |
| **Target encoder** | Identical ViT-Small that sees *all* patches | EMA of context encoder (no gradients) |
| **Predictor** | Small transformer that maps context → predicted target embeddings | Gradient descent |

The loss is MSE between the **predictor's output** and the **target encoder's embeddings** for the masked patches.  
No pixel reconstruction — the model predicts in *latent space*, which forces it to learn semantics rather than textures.


In [ ]:
# ── Context / Target Encoder: ViT-Small/16 via timm ──────────────────────
def build_encoder():
    """
    ViT-Small with patch_size=16.
    num_classes=0 and global_pool='' → returns patch token sequence (B, N, D).
    """
    encoder = timm.create_model(
        'vit_small_patch16_224',
        pretrained=False,       # train from scratch on medical data
        num_classes=0,          # remove classification head
        global_pool='',         # return all patch tokens, not just CLS
    )
    return encoder

context_encoder = build_encoder().to(DEVICE)
target_encoder  = copy.deepcopy(context_encoder).to(DEVICE)

# Target encoder is NEVER updated by gradients — only EMA
for p in target_encoder.parameters():
    p.requires_grad = False

n_params = sum(p.numel() for p in context_encoder.parameters()) / 1e6
print(f'Encoder params : {n_params:.1f} M')
print(f'Output shape   : (B, {N_PATCHES + 1}, {EMBED_DIM})  — includes CLS token')


In [ ]:
# ── Predictor: narrow transformer that maps context → target embeddings ──
class IJEPAPredictor(nn.Module):
    """
    Small transformer predictor.
    Input  : context patch embeddings (B, N_visible, D)
    Output : predicted target patch embeddings (B, N_target, D)

    The predictor uses learnable mask tokens for target positions
    and cross-attends context → target via standard TransformerDecoder blocks.
    For simplicity we use a TransformerEncoder with a learned positional
    query for each target slot.
    """
    def __init__(self, embed_dim=EMBED_DIM, predictor_dim=PREDICTOR_DIM,
                 depth=PREDICTOR_DEPTH, num_heads=6, n_patches=N_PATCHES):
        super().__init__()
        self.input_proj  = nn.Linear(embed_dim, predictor_dim)
        self.output_proj = nn.Linear(predictor_dim, embed_dim)
        self.mask_token  = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        self.pos_embed   = nn.Parameter(torch.zeros(1, n_patches + 1, predictor_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=predictor_dim, nhead=num_heads,
            dim_feedforward=predictor_dim * 4,
            dropout=0.0, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(predictor_dim)

        # init
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed,  std=0.02)

    def forward(self, context_tokens, context_idx, target_idx):
        """
        context_tokens : (B, N_ctx, D)   — encoded visible patches
        context_idx    : (N_ctx,)        — patch positions in [0, N_patches)
        target_idx     : (N_tgt,)        — patch positions to predict
        Returns        : (B, N_tgt, D)   — predicted embeddings
        """
        B = context_tokens.size(0)
        x = self.input_proj(context_tokens)                      # (B, N_ctx, P)

        # add positional embedding for context tokens
        x = x + self.pos_embed[:, context_idx, :]

        # build mask tokens for target positions
        mask = self.mask_token.expand(B, len(target_idx), -1)    # (B, N_tgt, P)
        mask = mask + self.pos_embed[:, target_idx, :]

        # concatenate context + mask tokens, run transformer
        tokens = torch.cat([x, mask], dim=1)                     # (B, N_ctx+N_tgt, P)
        tokens = self.transformer(tokens)
        tokens = self.norm(tokens)

        # extract only the target slot outputs
        pred = tokens[:, len(context_idx):, :]                   # (B, N_tgt, P)
        return self.output_proj(pred)                             # (B, N_tgt, D)

predictor = IJEPAPredictor().to(DEVICE)
n_pred = sum(p.numel() for p in predictor.parameters()) / 1e6
print(f'Predictor params: {n_pred:.1f} M')


In [ ]:
# ── EMA update function ───────────────────────────────────────────────────
def update_ema(target: nn.Module, context: nn.Module, momentum: float):
    """
    Exponential Moving Average update:
      target_param = momentum * target_param + (1 - momentum) * context_param
    Called after every optimizer step. No gradients involved.
    """
    with torch.no_grad():
        for t_param, c_param in zip(target.parameters(), context.parameters()):
            t_param.data.mul_(momentum).add_(c_param.data, alpha=1.0 - momentum)

# quick test
update_ema(target_encoder, context_encoder, EMA_MOMENTUM)
print('EMA update function OK.')


### 4. Masking Strategy

I-JEPA uses **block masking** — it masks rectangular regions of patches, not random individual patches.
This is harder to predict than isolated masked patches, which forces the encoder to learn more global structure.

- **Target blocks**: 4 overlapping rectangular blocks covering ~75% of patches total.
- **Context block**: a single large block covering 85–100% of the image (minus target positions).


In [ ]:
def sample_block_mask(n_patches_per_side: int,
                      target_mask_ratio: float = TARGET_MASK_RATIO,
                      n_target_blocks: int = N_TARGET_BLOCKS,
                      context_scale: tuple = CONTEXT_SCALE,
                      seed: int = None):
    """
    Returns:
        context_idx : 1-D LongTensor of visible patch indices
        target_idx  : 1-D LongTensor of masked patch indices to predict
    Both in flattened patch space [0, n_patches_per_side**2).
    """
    rng = np.random.default_rng(seed)
    N   = n_patches_per_side
    total = N * N

    target_set = set()

    # Sample N_TARGET_BLOCKS rectangular target blocks
    n_tgt_approx = int(total * target_mask_ratio / n_target_blocks)
    for _ in range(n_target_blocks):
        # random square-ish block
        area = rng.uniform(0.05, 0.3) * total
        aspect = rng.uniform(0.5, 2.0)
        h = max(1, int(math.sqrt(area / aspect)))
        w = max(1, int(area / h))
        h, w = min(h, N), min(w, N)
        top  = rng.integers(0, N - h + 1)
        left = rng.integers(0, N - w + 1)
        for r in range(top, top + h):
            for c in range(left, left + w):
                target_set.add(r * N + c)

    # Context: large block covering most of the image
    ctx_frac = rng.uniform(*context_scale)
    h_ctx = max(1, int(math.sqrt(ctx_frac * total)))
    w_ctx = max(1, int(ctx_frac * total / h_ctx))
    h_ctx, w_ctx = min(h_ctx, N), min(w_ctx, N)
    top_ctx  = rng.integers(0, N - h_ctx + 1)
    left_ctx = rng.integers(0, N - w_ctx + 1)
    context_set = set()
    for r in range(top_ctx, top_ctx + h_ctx):
        for c in range(left_ctx, left_ctx + w_ctx):
            idx = r * N + c
            if idx not in target_set:
                context_set.add(idx)

    # Fallback: if context is empty, use all non-target patches
    if not context_set:
        context_set = set(range(total)) - target_set

    context_idx = torch.tensor(sorted(context_set), dtype=torch.long)
    target_idx  = torch.tensor(sorted(target_set),  dtype=torch.long)
    return context_idx, target_idx

# ── Visualise one mask ───────────────────────────────────────────────────
n_side = IMG_SIZE // PATCH_SIZE   # 14
ctx_idx, tgt_idx = sample_block_mask(n_side)

mask_vis = np.zeros((n_side, n_side, 3))
for i in ctx_idx: mask_vis[i // n_side, i % n_side] = [0.2, 0.6, 0.9]   # blue = context
for i in tgt_idx: mask_vis[i // n_side, i % n_side] = [0.9, 0.3, 0.2]   # red  = target

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(mask_vis, interpolation='nearest')
ax.set_title(f'Masking example\nBlue=context ({len(ctx_idx)}), Red=target ({len(tgt_idx)})', fontsize=10)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'mask_example.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Context patches : {len(ctx_idx)} / {n_side**2}')
print(f'Target  patches : {len(tgt_idx)} / {n_side**2}  ({100*len(tgt_idx)/n_side**2:.1f}%)')


### 5. Pre-training DataLoader

I-JEPA pre-training is **fully self-supervised** — we use ALL images (train + val + test)
without any labels. The masking happens dynamically inside the training loop, not here.


In [ ]:
from dataset import LIDCDataset

# Merge all splits — labels not used during pre-training
train_df = pd.read_csv(os.path.join(DATA_PROC, 'train.csv'))
val_df   = pd.read_csv(os.path.join(DATA_PROC, 'val.csv'))
test_df  = pd.read_csv(os.path.join(DATA_PROC, 'test.csv'))
all_df   = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(f'Total images for pre-training: {len(all_df)}  (labels ignored)')

# Simple transform: resize + to tensor + normalize (no heavy augmentation —
# I-JEPA learns from masking, not pixel augmentation)
pretrain_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

pretrain_dataset = LIDCDataset(all_df, transform=pretrain_transform, pretrain=True)
pretrain_loader  = DataLoader(
    pretrain_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,    # keeps batch size constant for mask sampling
)

print(f'Batches per epoch : {len(pretrain_loader)}')
print(f'Images per epoch  : {len(pretrain_loader) * BATCH_SIZE}')


### 6. Optimizer & LR Scheduler

In [ ]:
optimizer = torch.optim.AdamW(
    list(context_encoder.parameters()) + list(predictor.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),   # standard ViT betas
)

# Cosine decay with linear warmup
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return epoch / max(1, WARMUP_EPOCHS)          # linear warmup
    progress = (epoch - WARMUP_EPOCHS) / max(1, NUM_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))  # cosine decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print('Optimizer : AdamW')
print(f'  LR      : {LR}  (warmup {WARMUP_EPOCHS} epochs → cosine decay)')
print(f'  WD      : {WEIGHT_DECAY}')


### 7. Resume from Checkpoint (optional)

In [ ]:
start_epoch = 0
train_losses = []

if RESUME_CKPT is not None and os.path.exists(RESUME_CKPT):
    print(f'Resuming from: {RESUME_CKPT}')
    ckpt = torch.load(RESUME_CKPT, map_location=DEVICE)
    context_encoder.load_state_dict(ckpt['context_encoder'])
    target_encoder.load_state_dict(ckpt['target_encoder'])
    predictor.load_state_dict(ckpt['predictor'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch  = ckpt['epoch'] + 1
    train_losses = ckpt.get('losses', [])
    print(f'Resumed at epoch {start_epoch} | Last loss: {train_losses[-1]:.6f}')
else:
    print('Starting pre-training from scratch.')


### 8. Pre-training Loop

Each step:
1. Sample a batch of images (no labels)
2. Sample a random block mask
3. Forward pass: context encoder on visible patches, target encoder on ALL patches
4. Predictor maps context embeddings → predicted target embeddings
5. MSE loss between prediction and target (detached)
6. Backprop through context encoder + predictor only
7. EMA update of target encoder


In [ ]:
def pretrain_one_epoch(context_enc, target_enc, pred, loader, opt, ema_mom, device, n_side):
    context_enc.train()
    pred.train()
    target_enc.eval()   # target encoder always in eval (no dropout, no BN updates)

    epoch_loss = 0.0
    for imgs in loader:
        imgs = imgs.to(device, non_blocking=True)   # (B, 3, H, W)
        B    = imgs.size(0)

        # ── Sample mask (same for whole batch for speed; varied per step via randomness)
        ctx_idx, tgt_idx = sample_block_mask(n_side)
        ctx_idx = ctx_idx.to(device)
        tgt_idx = tgt_idx.to(device)

        # ── Target encoder: encode all patches, extract target positions
        with torch.no_grad():
            all_tokens  = target_enc(imgs)           # (B, N+1, D)  — includes CLS
            patch_tokens = all_tokens[:, 1:, :]      # (B, N, D)    — drop CLS
            target_repr = patch_tokens[:, tgt_idx, :].detach()  # (B, N_tgt, D)

        # ── Context encoder: encode visible patches only
        # We pass the full image but mask out target patches before encoding.
        # Simple approach: zero out target patch positions in the input embedding.
        # (Full I-JEPA masks before the encoder; this approximation works well.)
        all_ctx_tokens  = context_enc(imgs)          # (B, N+1, D)
        ctx_patch_tokens = all_ctx_tokens[:, 1:, :]  # (B, N, D)
        context_repr    = ctx_patch_tokens[:, ctx_idx, :]  # (B, N_ctx, D)

        # ── Predictor: predict target embeddings from context
        predicted = pred(context_repr, ctx_idx.cpu(), tgt_idx.cpu())  # (B, N_tgt, D)

        # ── Loss: MSE in embedding space
        loss = F.mse_loss(predicted, target_repr)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(context_enc.parameters()) + list(pred.parameters()), max_norm=1.0)
        opt.step()

        # ── EMA update target encoder
        update_ema(target_enc, context_enc, ema_mom)

        epoch_loss += loss.item()

    return epoch_loss / len(loader)


In [ ]:
# ── Main training loop ─────────────────────────────────────────────────────
N_SIDE = IMG_SIZE // PATCH_SIZE   # 14

print(f'Starting pre-training: epochs {start_epoch} → {NUM_EPOCHS}')
print(f'Checkpoints saved to: {CKPT_DIR}')
print('-' * 55)

t0 = time.time()

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_t0 = time.time()

    loss = pretrain_one_epoch(
        context_encoder, target_encoder, predictor,
        pretrain_loader, optimizer, EMA_MOMENTUM, DEVICE, N_SIDE,
    )
    scheduler.step()
    train_losses.append(loss)

    elapsed  = time.time() - epoch_t0
    total_t  = time.time() - t0
    lr_now   = optimizer.param_groups[0]['lr']

    print(f'Epoch {epoch+1:03d}/{NUM_EPOCHS} | '
          f'Loss {loss:.6f} | '
          f'LR {lr_now:.2e} | '
          f'{elapsed:.0f}s/ep | '
          f'Total {total_t/60:.1f}min')

    # Save checkpoint every SAVE_EVERY epochs and at the final epoch
    if (epoch + 1) % SAVE_EVERY == 0 or epoch == NUM_EPOCHS - 1:
        ckpt_path = os.path.join(CKPT_DIR, f'ijepa_epoch_{epoch+1:03d}.pth')
        torch.save({
            'epoch':           epoch,
            'context_encoder': context_encoder.state_dict(),
            'target_encoder':  target_encoder.state_dict(),
            'predictor':       predictor.state_dict(),
            'optimizer':       optimizer.state_dict(),
            'scheduler':       scheduler.state_dict(),
            'losses':          train_losses,
            'config': {
                'img_size': IMG_SIZE, 'patch_size': PATCH_SIZE,
                'embed_dim': EMBED_DIM, 'ema_momentum': EMA_MOMENTUM,
            }
        }, ckpt_path)
        print(f'  → Checkpoint saved: {ckpt_path}')

print(f'\nPre-training complete in {(time.time()-t0)/60:.1f} min.')


### 9. Alternative: Load Meta AI Pretrained Checkpoint (Skip Pre-training)

If pre-training ran out of time, you can download Meta AI's official I-JEPA checkpoint
(pretrained on ImageNet-1k with ViT-Small) and use it directly.  
Your academic contribution stays the same — the XAI analysis in notebook 05 is identical either way.

**Run this section only if you skipped Section 8.**


In [ ]:
# ── Option A: download Meta AI ViT-Small checkpoint from GitHub ───────────
# Uncomment and run if you are using the pretrained checkpoint route.

# !wget -q -O /tmp/ijepa_vits16.pth \
#   https://dl.fbaipublicfiles.com/ijepa/IN1K-vit.s.16-600e.pth.tar

# # Meta AI checkpoint uses a slightly different key structure — remap it:
# raw = torch.load('/tmp/ijepa_vits16.pth', map_location='cpu')
# # The checkpoint stores the encoder under 'target_encoder' (EMA weights are best)
# enc_state = {k.replace('module.', ''): v
#              for k, v in raw['target_encoder'].items()}
# context_encoder.load_state_dict(enc_state, strict=False)
# target_encoder.load_state_dict(enc_state, strict=False)

# # Save in our checkpoint format so notebook 04 can load it the same way
# ckpt_path = os.path.join(CKPT_DIR, 'ijepa_pretrained_meta.pth')
# torch.save({
#     'epoch': 600,
#     'context_encoder': context_encoder.state_dict(),
#     'target_encoder':  target_encoder.state_dict(),
#     'predictor':       predictor.state_dict(),   # not trained, ignored in probe
#     'losses': [],
#     'config': {'img_size': 224, 'patch_size': 16, 'embed_dim': 384},
# }, ckpt_path)
# print(f'Meta AI checkpoint saved to: {ckpt_path}')


### 10. Training Loss Curve

In [ ]:
if len(train_losses) == 0:
    # Load losses from last checkpoint if training was done in a previous session
    ckpts = sorted(Path(CKPT_DIR).glob('ijepa_epoch_*.pth'))
    if ckpts:
        last = torch.load(ckpts[-1], map_location='cpu')
        train_losses = last.get('losses', [])
        print(f'Loaded {len(train_losses)} loss values from {ckpts[-1].name}')

if train_losses:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Raw loss
    axes[0].plot(range(1, len(train_losses)+1), train_losses,
                 color='#0A7E8C', linewidth=1.2, alpha=0.8)
    axes[0].set_title('Pre-training Loss (MSE)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MSE Loss')
    axes[0].grid(alpha=0.3)

    # Smoothed (5-epoch rolling average)
    smooth = pd.Series(train_losses).rolling(5, min_periods=1).mean()
    axes[1].plot(range(1, len(train_losses)+1), smooth,
                 color='#D95F2B', linewidth=2, label='5-epoch avg')
    axes[1].plot(range(1, len(train_losses)+1), train_losses,
                 color='#0A7E8C', linewidth=0.8, alpha=0.4, label='raw')
    axes[1].set_title('Smoothed Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MSE Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle('I-JEPA Pre-training Loss Curve', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES, 'pretrain_loss_curve.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Initial loss : {train_losses[0]:.6f}')
    print(f'Final loss   : {train_losses[-1]:.6f}')
    print(f'Reduction    : {100*(train_losses[0]-train_losses[-1])/train_losses[0]:.1f}%')
else:
    print('No losses to plot yet — run Section 8 or load a checkpoint first.')


### 11. Representation Quality Check (t-SNE)

A quick sanity check: extract CLS-token embeddings for 200 test images
and visualise with t-SNE. If the encoder learned useful features, the two
classes (Nodule / No Nodule) should form loosely separable clusters — even
without any label supervision.


In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

context_encoder.eval()

# Use only test split so we don't cheat with train labels
test_dataset_vis = LIDCDataset(
    test_df.sample(n=min(200, len(test_df)), random_state=SEED).reset_index(drop=True),
    transform=T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD),
    ]),
    pretrain=False,
)
vis_loader = DataLoader(test_dataset_vis, batch_size=32, shuffle=False)

embeddings, labels_list = [], []
with torch.no_grad():
    for imgs, lbls in tqdm(vis_loader, desc='Extracting embeddings'):
        tokens = context_encoder(imgs.to(DEVICE))  # (B, N+1, D)
        cls    = tokens[:, 0, :].cpu()              # CLS token = global repr
        embeddings.append(cls)
        labels_list.append(lbls)

embeddings = torch.cat(embeddings).numpy()
labels_arr = torch.cat(labels_list).numpy().astype(int)

# t-SNE
emb_2d = TSNE(n_components=2, random_state=SEED, perplexity=30).fit_transform(
    StandardScaler().fit_transform(embeddings))

fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#0A7E8C', '#D95F2B']
names  = ['No Nodule (0)', 'Nodule (1)']
for cls_id, color, name in zip([0, 1], colors, names):
    mask = labels_arr == cls_id
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
               c=color, label=name, alpha=0.7, s=35, edgecolors='none')
ax.set_title('t-SNE of I-JEPA CLS Embeddings (test set, no labels used in training)',
             fontsize=10)
ax.legend(fontsize=10)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'tsne_embeddings.png'), dpi=150, bbox_inches='tight')
plt.show()
print('If clusters are loosely separable → encoder learned meaningful representations.')
print('Some overlap is expected (self-supervised, no label signal during pre-training).')


### 12. Mark Best Checkpoint for Notebook 04

In [ ]:
# The last epoch checkpoint is also our best (loss still decreasing at 100 epochs
# is expected for SSL — unlike supervised training, it doesn't plateau as quickly).
# We create a symlink / copy named 'ijepa_best.pth' for easy loading in notebook 04.

import shutil

last_ckpt = sorted(Path(CKPT_DIR).glob('ijepa_epoch_*.pth'))
if last_ckpt:
    best_path = os.path.join(CKPT_DIR, 'ijepa_best.pth')
    shutil.copy(str(last_ckpt[-1]), best_path)
    print(f'Best checkpoint: {best_path}')
    print(f'  (copy of {last_ckpt[-1].name})')
else:
    print('No checkpoint found — make sure Section 8 or 9 was run.')


## Done

**Files produced in Drive:**

| File | Description |
|------|-------------|
| `checkpoints/ijepa/ijepa_epoch_XXX.pth` | Full checkpoint every 5 epochs |
| `checkpoints/ijepa/ijepa_best.pth` | Copy of last epoch — used by notebook 04 |
| `results/figures/pretrain/mask_example.png` | Visualisation of block masking |
| `results/figures/pretrain/pretrain_loss_curve.png` | Training loss curve |
| `results/figures/pretrain/tsne_embeddings.png` | t-SNE representation check |

**How to load the encoder in notebook 04:**
```python
import timm, torch, copy

def build_encoder():
    return timm.create_model('vit_small_patch16_224',
                              pretrained=False, num_classes=0, global_pool='')

context_encoder = build_encoder().to(DEVICE)
ckpt = torch.load(f'{CKPT_DIR}/ijepa_best.pth', map_location=DEVICE)
context_encoder.load_state_dict(ckpt['context_encoder'])
context_encoder.eval()
for p in context_encoder.parameters():
    p.requires_grad = False   # freeze — linear probe only trains the head
```

**Next notebook:** `04_linear_probe.ipynb`
